# tp-05g 新しい特徴量（追い切り評価・厩舎コメント・血統×条件）の効果

**実行場所: ホスト PC（stg）。** 採否は valid 2022〜2024 だけで決める（test 2025 は既に見たので使わない）。基準は tp-05d の調整済み LightGBM（`lgb_tuned`）。

| 特徴量群 | 内容 |
|---|---|
| extras | 追い切り評価（A〜D → 4〜1）と、レース内の順位。厩舎コメントの有無・長さ・前向き／後ろ向きの語の数。いずれもレース前に公開される情報 |
| ped | 父・母父 × (芝ダ×距離帯)・(芝ダ×馬場状態)・(芝ダ×競馬場) の産駒の勝率・複勝率（経験ベイズで縮約）、父の産駒の脚質傾向（pos_early）と上がりの速さ（l3f_dev）。すべて、そのレースより厳密に過去の行だけで集計 |
| extras+ped | 両方 |

段階 1: LightGBM で 3 案を比べる。段階 2: 採用基準を満たした特徴量群があれば、XGBoost・NN（調整後パラメータ）にも入れて OOF を保存し、tp-05h のアンサンブルに回す。
採否: p_win の LogLoss の差の 95% 信頼区間の上限が 0 未満で、連対・複勝が悪化しない（3 fold × 3 seed）。

In [ ]:
import os, sys
from pathlib import Path


def _find_root():
    p = Path.cwd().resolve()
    for q in [p, *p.parents]:
        if (q / "AGENTS.md").exists() and (q / "src").is_dir():
            return q
    raise RuntimeError("リポジトリルートが見つかりません")


REPO_ROOT = _find_root()
os.chdir(REPO_ROOT)
os.environ.setdefault("POLARS_MAX_THREADS", "6")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
from src.utils.project_env import load_project_dotenv
load_project_dotenv()

In [ ]:
import json, subprocess, gc
import numpy as np
import pandas as pd
import mlflow
import psutil

from src.pipeline.mlflow.train_utils import require_tracking_server
from src.pipeline.features import pedigree_context
from src.pipeline.models import tp05_data, zoo, zoo_runner, win_models as wm
from src.pipeline.models.compute import detect_compute

WORK = REPO_ROOT / "data/local/modeling/training_pipeline"
OOF_DIR = WORK / "oof/model_oof"
OUT = WORK / "improve_g"
OUT.mkdir(parents=True, exist_ok=True)
THREADS = detect_compute(reserve_cores=6).threads
SEEDS = [0, 1, 2]
mlflow.set_experiment("keiba-tp-model-zoo")
require_tracking_server()
GIT_COMMIT = subprocess.run(["git", "rev-parse", "--short", "HEAD"], capture_output=True, text=True, cwd=REPO_ROOT).stdout.strip()
D = tp05_data.load_dataset(WORK)
df, meta, folds = D["df"], D["meta"], D["folds"]
best = json.loads((WORK / "hpo/best_params.json").read_text(encoding="utf-8"))["best"]
X1 = df[D["FEATS"]["base+S+C1"]]

# extras: 追い切り評価・厩舎コメント
ex = pd.read_parquet(WORK / "features/prerace_extras.parquet")
ex = ex.drop(columns=[c for c in ("comment_result_words",) if c in ex.columns])
E = df[["race_id", "horse_id"]].merge(ex, on=["race_id", "horse_id"], how="left")
E.index = df.index
E["has_comment"] = E["has_comment"].fillna(0.0)
E["oikiri_eval_rank"] = E.groupby("race_id")["oikiri_eval"].rank(ascending=False, method="average")
E["oikiri_eval_gap"] = E["oikiri_eval"] - E.groupby("race_id")["oikiri_eval"].transform("max")
EXTRAS = [c for c in E.columns if c not in ("race_id", "horse_id")]
print("extras 列:", EXTRAS, "| 被覆率:", {c: round(float(E[c].notna().mean()), 3) for c in EXTRAS[:4]})

# ped: 父・母父 × 条件
labels = pd.read_parquet(WORK / "oof/stage_labels.parquet")
P = pedigree_context.build(df, labels)
PED = list(P.columns)
print(f"ped 列 {len(PED)} 本 | 欠損率の平均 {float(P.isna().mean().mean()):.3f}")
SETS = {"extras": EXTRAS, "ped": PED, "extras+ped": EXTRAS + PED}
XS = {k: pd.concat([X1, E[v] if k == "extras" else (P if k == "ped" else pd.concat([E[EXTRAS], P], axis=1))], axis=1) for k, v in SETS.items()}
print({k: v.shape[1] for k, v in XS.items()}, f"| mem {psutil.Process().memory_info().rss / 1e9:.1f}GB")

In [ ]:
lgb_p = zoo.tuned_params("lgb", best["lgb"]["params"])
RES, OOFS = [], {}
for name, X in XS.items():
    nm = f"lgb_{name.replace('+', '_')}"
    with mlflow.start_run(run_name=f"tp-05g-{nm}", tags={"notebook": "tp-05g", "git_commit": GIT_COMMIT, "variant": nm, "base": "lgb_tuned"}):
        res, OOFS[nm] = zoo_runner.run_oof(nm, "lgb_binary", X, meta, folds, SEEDS, THREADS, params=dict(lgb_p), out_dir=OOF_DIR)
        num = [c for c in res.columns if c not in ("model", "fold")]
        mlflow.log_params({"n_features": X.shape[1], "feature_group": name})
        mlflow.log_metrics({f"mean_{c}": float(res[c].mean()) for c in num})
    print(f"  {nm:18s} win {res['logloss_win'].mean():.5f} top2 {res['logloss_top2'].mean():.5f} top3 {res['logloss_top3'].mean():.5f} ({res.attrs['seconds']:.0f}s)")
    RES.append(res); gc.collect()

In [ ]:
key = ["race_id", "horse_id"]
fin_map = meta.set_index(key)["finish_position"]


def compare(new_name, ref_name, new_oof=None):
    b = pd.read_parquet(OOF_DIR / f"{ref_name}.parquet")
    m = b.merge(new_oof if new_oof is not None else OOFS[new_name], on=key, suffixes=("_b", "_v"))
    fin = fin_map.loc[list(zip(m["race_id"], m["horse_id"]))].to_numpy()
    out = {"variant": new_name, "基準": ref_name}
    for tgt, k in (("win", 1), ("top2", 2), ("top3", 3)):
        y = (fin <= k).astype(float)
        bb = wm.paired_bootstrap_diff(wm.per_race_loss(m["race_id"].to_numpy(), m[f"p_{tgt}_b"].to_numpy(), y), wm.per_race_loss(m["race_id"].to_numpy(), m[f"p_{tgt}_v"].to_numpy(), y), n_boot=500)
        out[f"{tgt}_base"], out[f"{tgt}_new"] = wm.binary_logloss(m[f"p_{tgt}_b"].to_numpy(), y), wm.binary_logloss(m[f"p_{tgt}_v"].to_numpy(), y)
        out[f"Δ{tgt}"], out[f"{tgt}_hi"] = bb["diff"], bb["hi"]
    out["採用"] = bool(out["win_hi"] < 0 and out["Δtop2"] <= 0.0002 and out["Δtop3"] <= 0.0002)
    return out


cmp1 = pd.DataFrame([compare(n, "lgb_tuned") for n in OOFS]).set_index("variant")
display(cmp1.round(5)); cmp1.to_csv(OUT / "stage1_vs_lgb_tuned.csv")
ok = cmp1[cmp1["採用"]].sort_values("Δwin")
BEST_SET = ok.index[0].replace("lgb_", "").replace("_", "+") if len(ok) else None
print("段階 1 の結果: 採用する特徴量群 =", BEST_SET)

In [ ]:
# 段階 2: 採用基準を満たした特徴量群があれば、XGBoost・NN（調整後パラメータ）にも入れて OOF を保存する
STAGE2 = []
if BEST_SET:
    X = XS[BEST_SET]
    IDS = None
    runs = {f"xgb_{BEST_SET.replace('+', '_')}": ("xgb_binary", zoo.tuned_params("xgb", best["xgb"]["params"]), "xgb_tuned"),
            f"nn_{BEST_SET.replace('+', '_')}": ("nn", zoo.tuned_params("nn", best["nn"]["params"]), "nn_tuned")}
    for nm, (recipe, params, ref) in runs.items():
        with mlflow.start_run(run_name=f"tp-05g-{nm}", tags={"notebook": "tp-05g", "git_commit": GIT_COMMIT, "variant": nm, "base": ref}):
            res, oof = zoo_runner.run_oof(nm, recipe, X, meta, folds, SEEDS, THREADS, params=dict(params), out_dir=OOF_DIR)
            mlflow.log_metrics({f"mean_{c}": float(res[c].mean()) for c in res.columns if c not in ("model", "fold")})
        print(f"  {nm:18s} win {res['logloss_win'].mean():.5f} top2 {res['logloss_top2'].mean():.5f} top3 {res['logloss_top3'].mean():.5f} ({res.attrs['seconds']:.0f}s)")
        STAGE2.append(compare(nm, ref, oof))
        gc.collect()
    cmp2 = pd.DataFrame(STAGE2).set_index("variant")
    display(cmp2.round(5)); cmp2.to_csv(OUT / "stage2_vs_tuned.csv")
(OUT / "report.json").write_text(json.dumps({"git_commit": GIT_COMMIT, "best_set": BEST_SET, "stage1": cmp1.round(6).reset_index().to_dict("records"),
                                             "stage2": STAGE2, "extras_cols": EXTRAS, "ped_cols": PED}, ensure_ascii=False, indent=1, default=str), encoding="utf-8")
# 特徴量の寄与（採用した群があれば LightGBM で重要度を確認）
if BEST_SET:
    import lightgbm as lgb
    f = folds[2]
    y = (meta["finish_position"].to_numpy() == 1).astype(int)
    Xb = XS[BEST_SET]
    cats = [c for c in Xb.columns if str(Xb[c].dtype) == "category"]
    prm = dict({"objective": "binary", "verbosity": -1, "num_threads": THREADS, "seed": 0, "bagging_freq": 1}, **lgb_p["lgb"])
    bst = lgb.train(prm, lgb.Dataset(Xb.iloc[f.train_index], label=y[f.train_index], categorical_feature=cats), num_boost_round=lgb_p["n_rounds"])
    g = pd.Series(bst.feature_importance("gain"), index=Xb.columns); g = g / g.sum()
    new_cols = [c for c in Xb.columns if c in set(EXTRAS + PED)]
    print("新しい特徴量の寄与合計:", round(float(g[new_cols].sum()) * 100, 2), "%"); display(g[new_cols].sort_values(ascending=False).head(12).round(4))